# Kubeflow Pipeline

Train the YOLO license-plate model as a pipeline.

**Covered here:** `fetch_data`.

References:

- <https://www.kubeflow.org/docs/components/pipelines/getting-started/>
- <https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/connect-api/>

## Environment

In [ ]:
# pip install
%pip install -q -U kfp

## Define

`fetch_data` downloads the DVC-tracked raw dataset from S3.

`dvc_dir_hash` is the md5 in `data/raw.dvc`. It addresses a `.dir` manifest in
S3 listing `{md5, relpath}` for every file, so one hash pins the whole dataset
version.

S3 needs no credentials here: the pod runs as `default-editor`, which carries
the S3 role by EKS Pod Identity (`infra/60-s3-iam.tf`).

In [ ]:
from kfp import dsl
from kfp.dsl import Dataset, Output


@dsl.component(base_image="python:3.12", packages_to_install=["boto3"])
def fetch_data(
    bucket: str,
    dvc_dir_hash: str,
    region: str,
    raw: Output[Dataset],
):
    import json
    from concurrent.futures import ThreadPoolExecutor
    from pathlib import Path

    import boto3

    s3 = boto3.client("s3", region_name=region)

    def dvc_key(md5: str) -> str:
        # DVC shards its content-addressed store by the first two hex chars
        return "dvcstore/files/md5/" + md5[:2] + "/" + md5[2:]

    # the .dir object lists {"md5": ..., "relpath": ...} for every file
    manifest = json.loads(
        s3.get_object(Bucket=bucket, Key=dvc_key(dvc_dir_hash))["Body"].read()
    )

    root = Path(raw.path)
    root.mkdir(parents=True, exist_ok=True)

    def fetch(entry):
        target = root / entry["relpath"]
        target.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, dvc_key(entry["md5"]), str(target))

    # 1100+ small objects: latency-bound, not bandwidth-bound
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(fetch, manifest))

    images = [p for p in root.iterdir()
              if p.suffix.lower() in {".jpeg", ".jpg", ".png"}]
    if not images:
        raise RuntimeError("no images restored -- check the dvc_dir_hash")

    raw.metadata["files"] = len(manifest)
    raw.metadata["images"] = len(images)

    print("fetched", len(manifest), "files /", len(images), "images")


@dsl.pipeline
def yolo_pipeline(
    bucket: str = "kubeflow-yolo-dev-099139718958",
    dvc_dir_hash: str = "0e94102a7a6b4424a0f1292c2f221072.dir",
    region: str = "ca-central-1",
):
    fetch_data(bucket=bucket, dvc_dir_hash=dvc_dir_hash, region=region)

## Compile

Produces a self-contained pipeline yaml. Needs no cluster.

In [ ]:
from kfp import compiler

compiler.Compiler().compile(yolo_pipeline, "yolo_pipeline.yaml")

## Connect

Inside the cluster `kfp.Client()` needs no arguments: it reads the token from
`KF_PIPELINES_SA_TOKEN_PATH` and defaults to
`http://ml-pipeline-ui.kubeflow.svc.cluster.local`.

The token volume comes from a `PodDefault` in this profile namespace.

In [ ]:
import kfp

kfp_client = kfp.Client()

# test the client by listing experiments
experiments = kfp_client.list_experiments(namespace="kubeflow-user-example-com")
print(experiments)

## Run

In [ ]:
run = kfp_client.create_run_from_pipeline_package(
    "yolo_pipeline.yaml",
    arguments={},
)

print(run.run_id)